# PetGuard AI | Final Application and Evaluation

**ITAI 1378 Computer Vision and AI**

Process fixed-camera video to detect dogs and cats, check a selected restricted zone, and export annotated video and violation records.

**Workflow:** A. Setup and processing | B. Export run | C. Prepare labels | D. Upload reviewed labels | E1–E4. Validate, evaluate, review, and download.

**Evaluation note:** Model predictions are not ground truth. Labels reviewed only from annotated output remain provisional until checked against original footage. Results from one short clip do not establish general accuracy. Colab runtime files are temporary, so download outputs before disconnecting.

## A1. Set Up the Project

Run once per new Colab runtime to clone the repository and install dependencies. An existing checkout is preserved and does not automatically receive newer commits.


In [1]:
from pathlib import Path
import subprocess, sys, os
REPO = Path('/content/PetGuard-AI')
URL = 'https://github.com/iamhuongthn/PetGuard-AI.git'
if not (REPO / '.git').is_dir():
    if REPO.exists():
        raise RuntimeError(f'{REPO} exists but is not a Git clone; inspect it before proceeding.')
    subprocess.run(['git','clone','--branch','master',URL,str(REPO)],check=True)
else:
    print('Existing repository retained; no destructive reset performed.')
os.chdir(REPO)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements.txt')],check=True)
assert (REPO/'src/app.py').is_file()
print('Repository ready:',REPO)

Repository ready: /content/PetGuard-AI


## A2. Process a Video

Launch Gradio, upload a fixed-camera MP4, select the zone type, draw at least three corners, and confirm the polygon. Click **Process Video** to generate annotated output and event records.

A violation is confirmed after five consecutive positive frames at the scene level. Individual pets are not tracked across frames. The temporary Gradio share link is not a permanent demo link.

In [2]:
import sys
sys.path.insert(0,str(REPO))
from src.app import build_app
app = build_app()
app.queue(default_concurrency_limit=1).launch(share=True, show_error=True, prevent_thread_lock=True)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8634f4b31345ff8c0b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## B. Save the Completed Video Run

Export the latest processing run as a ZIP containing annotated videos, `events.csv`, `frame_predictions.csv`, and `zone_config.json`. Each run has a unique ID. Download the ZIP before the Colab runtime disconnects.

In [4]:
from google.colab import files
from scripts.export_run import export
runs_dir = REPO/'results'/'runs'
runs = sorted((p for p in runs_dir.iterdir() if p.is_dir()), key=lambda p:p.stat().st_mtime) if runs_dir.exists() else []
assert runs, 'No runs found. Process a video in Gradio first.'
latest = runs[-1]
required = ['frame_predictions.csv','zone_config.json','events.csv']
assert all((latest/x).is_file() for x in required), f'Run {latest.name} is incomplete.'
run_zip = export(latest, f'/content/petguard_run_{latest.name}.zip')
print('Exported run:',latest.name)
files.download(str(run_zip))

Exported run: e7aee7a13672


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## C. Prepare a Frame-Labeling Template

Generate a CSV with up to 50 sampled frames from the latest run. Predicted zone statuses are copied automatically. Leave `expected_zone_status` and `pet_count_ground_truth` blank until the original video is reviewed.

In [5]:
import pandas as pd, numpy as np, json
RUN_DIR = latest  # Or set explicitly: REPO/'results'/'runs'/'your_run_id'
frames = pd.read_csv(RUN_DIR/'frame_predictions.csv')
config = json.loads((RUN_DIR/'zone_config.json').read_text())
assert {'frame','candidate_violation'}.issubset(frames.columns)
assert len(frames) > 0
n_samples = min(50,len(frames))
selected = frames.iloc[np.linspace(0,len(frames)-1,n_samples,dtype=int)].drop_duplicates(subset='frame')
mode = config.get('zone_mode','unknown')
labels = pd.DataFrame({
 'sample_id':[f'{RUN_DIR.name}_{int(f):05d}' for f in selected['frame']],
 'source':'real_video',
 'video_id':RUN_DIR.name,
 'frame_id':selected['frame'].astype(int).to_numpy(),
 'scenario':f'{mode}_zone',
 'pet_count_ground_truth':'',
 'expected_zone_status':'',
 'predicted_zone_status':selected['candidate_violation'].astype(bool).map({True:'VIOLATION',False:'SAFE'}).to_numpy(),
 'zone_mode':mode,
 'notes':''
})
template_path=Path(f'/content/zone_labels_TO_REVIEW_{RUN_DIR.name}.csv')
labels.to_csv(template_path,index=False)
print('Generated blank human-label template:',template_path,'Rows:',len(labels),'Mode:',mode)
files.download(str(template_path))

Generated blank human-label template: /content/zone_labels_TO_REVIEW_e7aee7a13672.csv Rows: 50 Mode: floor


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## D. Upload Reviewed Labels

Complete the template on a PC. Enter the actual pet count and `SAFE` or `VIOLATION` for each frame. Keep predictions unchanged and note uncertain cases. Upload the completed CSV to save it as `data/zone_labels_verified.csv`.

Labels based only on annotated video must remain marked provisional.


In [6]:
# ============================================================
# SECTION D — Upload completed zone labels from your PC
# ============================================================

from google.colab import files
from pathlib import Path
import pandas as pd

# 1. Select CSV from your computer
print("Select your completed zone-label CSV from your PC.")
uploaded = files.upload()

# 2. Validate uploaded file
csv_names = [
    name for name in uploaded
    if name.lower().endswith(".csv")
]

assert len(csv_names) == 1, (
    "Please upload exactly one CSV file."
)

uploaded_name = csv_names[0]

# 3. Read CSV directly from uploaded bytes
from io import BytesIO

df = pd.read_csv(BytesIO(uploaded[uploaded_name]))

# 4. Validate required columns
required_columns = [
    "sample_id",
    "frame_id",
    "expected_zone_status",
    "predicted_zone_status"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

assert not missing, f"Missing columns: {missing}"
assert not df.empty, "CSV contains no evaluation rows."

# 5. Validate ground-truth and predicted labels
for col in ["expected_zone_status", "predicted_zone_status"]:
    assert df[col].notna().all(), (
        f"Column '{col}' contains missing values."
    )

    df[col] = df[col].astype(str).str.strip().str.upper()

    invalid = ~df[col].isin(["SAFE", "VIOLATION"])

    assert not invalid.any(), (
        f"Invalid values in {col}: "
        f"{df.loc[invalid, col].unique().tolist()}"
    )

assert not df["sample_id"].duplicated().any(), (
    "Duplicate sample_id values detected."
)

# 6. Save to PetGuard project directory
REPO = Path("/content/PetGuard-AI")

verified_path = REPO / "data" / "zone_labels_verified.csv"
verified_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(verified_path, index=False, encoding="utf-8")

# 7. Display confirmation
print("\n" + "=" * 55)
print("CSV UPLOAD SUCCESSFUL")
print("=" * 55)

print(f"Original filename : {uploaded_name}")
print(f"Saved filename    : {verified_path.name}")
print(f"Saved location    : {verified_path}")
print(f"Total samples     : {len(df)}")

print("\nGround-truth labels:")
print(df["expected_zone_status"].value_counts().to_string())

print("\nPredicted labels:")
print(df["predicted_zone_status"].value_counts().to_string())

print("\nPreview:")
display(df.head(10))

print("\nReady for Section E — Zone Evaluation.")

Select your completed zone-label CSV from your PC.


Saving zone_labels_verified_e7aee7a13672.csv to zone_labels_verified_e7aee7a13672.csv

CSV UPLOAD SUCCESSFUL
Original filename : zone_labels_verified_e7aee7a13672.csv
Saved filename    : zone_labels_verified.csv
Saved location    : /content/PetGuard-AI/data/zone_labels_verified.csv
Total samples     : 50

Ground-truth labels:
expected_zone_status
SAFE         41
VIOLATION     9

Predicted labels:
predicted_zone_status
SAFE         43
VIOLATION     7

Preview:


,sample_id,source,video_id,frame_id,scenario,pet_count_ground_truth,expected_zone_status,predicted_zone_status,zone_mode,notes
0,e7aee7a13672_00000,real_video,e7aee7a13672,0,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
1,e7aee7a13672_00006,real_video,e7aee7a13672,6,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
2,e7aee7a13672_00012,real_video,e7aee7a13672,12,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
3,e7aee7a13672_00019,real_video,e7aee7a13672,19,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
4,e7aee7a13672_00025,real_video,e7aee7a13672,25,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
5,e7aee7a13672_00032,real_video,e7aee7a13672,32,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
6,e7aee7a13672_00038,real_video,e7aee7a13672,38,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
7,e7aee7a13672_00045,real_video,e7aee7a13672,45,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
8,e7aee7a13672_00051,real_video,e7aee7a13672,51,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
9,e7aee7a13672_00058,real_video,e7aee7a13672,58,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...



Ready for Section E — Zone Evaluation.


## E1. Validate Labels

Check required columns, unique IDs, pet counts, and valid `SAFE` or `VIOLATION` values. Review label counts before evaluation. Structural validation does not verify label correctness.

In [7]:
# ============================================================
# SECTION E1 — Validate uploaded ground-truth labels
# ============================================================

import pandas as pd
from pathlib import Path

# 1. Load CSV saved by Section D
verified_path = Path(
    "/content/PetGuard-AI/data/zone_labels_verified.csv"
)

assert verified_path.exists(), (
    f"CSV not found: {verified_path}. Run Section D first."
)

df = pd.read_csv(verified_path)

# 2. Validate required columns
required = {
    "sample_id",
    "frame_id",
    "video_id",
    "pet_count_ground_truth",
    "expected_zone_status",
    "predicted_zone_status",
    "zone_mode"
}

missing = required - set(df.columns)
assert not missing, f"Missing columns: {missing}"

# 3. Validate samples
assert not df.empty, "Evaluation CSV is empty."

assert not df["sample_id"].duplicated().any(), (
    "Duplicate sample IDs detected."
)

assert df["frame_id"].notna().all(), (
    "Some frame IDs are missing."
)

assert df["video_id"].notna().all(), (
    "Some video IDs are missing."
)

# 4. Validate SAFE / VIOLATION labels
for col in [
    "expected_zone_status",
    "predicted_zone_status"
]:
    assert df[col].notna().all(), (
        f"Missing values in {col}"
    )

    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    assert df[col].isin(
        ["SAFE", "VIOLATION"]
    ).all(), f"Invalid labels in {col}"

# 5. Validate pet counts
pet_counts = pd.to_numeric(
    df["pet_count_ground_truth"],
    errors="coerce"
)

assert pet_counts.notna().all(), (
    "Missing or invalid pet counts."
)

assert (
    (pet_counts >= 0) &
    (pet_counts % 1 == 0)
).all(), "Pet counts must be non-negative integers."

# 6. Display summary
print("=" * 55)
print("PETGUARD AI — GROUND-TRUTH VALIDATION")
print("=" * 55)

print("Samples:", len(df))
print("Video runs:", df["video_id"].nunique())

print("\nActual labels:")
print(df["expected_zone_status"].value_counts())

print("\nPredicted labels:")
print(df["predicted_zone_status"].value_counts())

print("\nZone modes:")
print(df["zone_mode"].value_counts())

print("\nConfusion matrix preview:")
print(pd.crosstab(
    df["expected_zone_status"],
    df["predicted_zone_status"],
    rownames=["Actual"],
    colnames=["Predicted"]
))

print("\nFirst 10 rows:")
display(df.head(10))

print("\nVALIDATION PASSED")
print(
    "Note: Structural validation does not prove that "
    "ground-truth labels are correct. Verify against "
    "original unannotated footage."
)

PETGUARD AI — GROUND-TRUTH VALIDATION
Samples: 50
Video runs: 1

Actual labels:
expected_zone_status
SAFE         41
VIOLATION     9
Name: count, dtype: int64

Predicted labels:
predicted_zone_status
SAFE         43
VIOLATION     7
Name: count, dtype: int64

Zone modes:
zone_mode
floor    50
Name: count, dtype: int64

Confusion matrix preview:
Predicted  SAFE  VIOLATION
Actual                    
SAFE         41          0
VIOLATION     2          7

First 10 rows:


,sample_id,source,video_id,frame_id,scenario,pet_count_ground_truth,expected_zone_status,predicted_zone_status,zone_mode,notes
0,e7aee7a13672_00000,real_video,e7aee7a13672,0,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
1,e7aee7a13672_00006,real_video,e7aee7a13672,6,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
2,e7aee7a13672_00012,real_video,e7aee7a13672,12,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
3,e7aee7a13672_00019,real_video,e7aee7a13672,19,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
4,e7aee7a13672_00025,real_video,e7aee7a13672,25,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
5,e7aee7a13672_00032,real_video,e7aee7a13672,32,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
6,e7aee7a13672_00038,real_video,e7aee7a13672,38,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
7,e7aee7a13672_00045,real_video,e7aee7a13672,45,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
8,e7aee7a13672_00051,real_video,e7aee7a13672,51,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...
9,e7aee7a13672_00058,real_video,e7aee7a13672,58,floor_zone,2,SAFE,SAFE,floor,Two dogs visible outside the left restricted a...



VALIDATION PASSED
Note: Structural validation does not prove that ground-truth labels are correct. Verify against original unannotated footage.


## E2. Generate Evaluation Results

Compare expected and predicted frame-level zone statuses. Save accuracy, false violation rate, violation recall, confusion matrix, failure cases, and a summary in `results/metrics/`. These metrics do not measure confirmed-event accuracy.

In [8]:
# ============================================================
# SECTION E2 — Generate PetGuard Zone Evaluation Results
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

REPO = Path("/content/PetGuard-AI")
verified_path = REPO / "data" / "zone_labels_verified.csv"
metrics_dir = REPO / "results" / "metrics"

# 1. Verify input
assert verified_path.exists(), (
    f"Verified labels not found: {verified_path}. "
    "Please complete Sections D and E1 first."
)

# 2. Create output directory
metrics_dir.mkdir(parents=True, exist_ok=True)

# 3. Run evaluation from project root
print("=" * 55)
print("PETGUARD AI — RUNNING ZONE EVALUATION")
print("=" * 55)

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "scripts.evaluate_zone_labels",
        str(verified_path),
        "--out",
        str(metrics_dir)
    ],
    cwd=str(REPO),
    capture_output=True,
    text=True
)

# 4. Display execution results
if result.stdout:
    print(result.stdout)

if result.returncode != 0:
    print("Evaluation failed.")
    print(result.stderr)
    raise RuntimeError("PetGuard zone evaluation failed.")

# 5. Verify generated files
expected_files = [
    "zone_evaluation.csv",
    "zone_confusion_matrix.csv",
    "zone_failure_cases.csv",
    "zone_evaluation_summary.md"
]

missing = [
    name for name in expected_files
    if not (metrics_dir / name).is_file()
]

assert not missing, (
    f"Missing evaluation outputs: {missing}"
)

print("\nEVALUATION COMPLETED SUCCESSFULLY")
print("Output folder:", metrics_dir)

for name in expected_files:
    print("✓", name)

PETGUARD AI — RUNNING ZONE EVALUATION
{
  "samples": 50,
  "TP": 7,
  "TN": 41,
  "FP": 0,
  "FN": 2,
  "zone_accuracy": 0.96,
  "false_violation_rate": 0.0,
  "violation_recall": 0.7777777777777778
}


EVALUATION COMPLETED SUCCESSFULLY
Output folder: /content/PetGuard-AI/results/metrics
✓ zone_evaluation.csv
✓ zone_confusion_matrix.csv
✓ zone_failure_cases.csv
✓ zone_evaluation_summary.md


## E3. Review Results

Inspect the metrics, confusion matrix, and missed violations. Compare zone accuracy with the 90% target and false violation rate with the 10% maximum. Report limitations and keep annotated-video labels provisional until independently checked.



In [9]:
# ============================================================
# SECTION E3 — Review Evaluation Results
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

metrics_dir = Path(
    "/content/PetGuard-AI/results/metrics"
)

# 1. Load evaluation outputs
evaluation = pd.read_csv(
    metrics_dir / "zone_evaluation.csv"
)

confusion = pd.read_csv(
    metrics_dir / "zone_confusion_matrix.csv"
)

failures = pd.read_csv(
    metrics_dir / "zone_failure_cases.csv"
)

summary = (
    metrics_dir / "zone_evaluation_summary.md"
).read_text(encoding="utf-8")

# 2. Show evaluation metrics
print("=" * 55)
print("PETGUARD AI — FINAL ZONE EVALUATION")
print("=" * 55)

print("\n1. EVALUATION METRICS")
display(evaluation)

# 3. Show confusion matrix
print("\n2. CONFUSION MATRIX")
display(confusion)

# 4. Show failure cases
print("\n3. FAILURE CASES")

if failures.empty:
    print("No misclassified frames found.")
else:
    print(f"Total misclassified frames: {len(failures)}")
    display(failures)

# 5. Render Markdown summary
print("\n4. EVALUATION SUMMARY")
display(Markdown(summary))

# 6. Check against Midterm targets
row = evaluation.iloc[0]

accuracy = float(row["zone_accuracy"])
false_rate = float(row["false_violation_rate"])
recall = float(row["violation_recall"])

print("\n" + "=" * 55)
print("RESULTS VS MIDTERM TARGETS")
print("=" * 55)

print(
    f"Zone Accuracy: {accuracy:.2%} "
    f"(Target ≥90%) — "
    f"{'MET' if accuracy >= 0.90 else 'NOT MET'}"
)

print(
    f"False Violation Rate: {false_rate:.2%} "
    f"(Target ≤10%) — "
    f"{'MET' if false_rate <= 0.10 else 'NOT MET'}"
)

print(
    f"Violation Recall: {recall:.2%} "
    "(Additional diagnostic metric)"
)

print("\nNote: Results are calculated from the supplied labels.")
print(
    "Labels based on annotated video remain provisional "
    "until checked against original unannotated footage."
)

PETGUARD AI — FINAL ZONE EVALUATION

1. EVALUATION METRICS


,samples,TP,TN,FP,FN,zone_accuracy,false_violation_rate,violation_recall
0,50,7,41,0,2,0.96,0.0,0.777778



2. CONFUSION MATRIX


,Actual,SAFE,VIOLATION
0,SAFE,41,0
1,VIOLATION,2,7



3. FAILURE CASES
Total misclassified frames: 2


,sample_id,source,video_id,frame_id,scenario,pet_count_ground_truth,expected_zone_status,predicted_zone_status,zone_mode,notes
0,e7aee7a13672_00188,real_video,e7aee7a13672,188,floor_zone,2,VIOLATION,SAFE,floor,Boundary ambiguous; provisional label from ann...
1,e7aee7a13672_00194,real_video,e7aee7a13672,194,floor_zone,2,VIOLATION,SAFE,floor,Boundary ambiguous; provisional label from ann...



4. EVALUATION SUMMARY


# PetGuard Zone Evaluation (human-labeled)

Samples: 50

Accuracy: 96.00%

False violation rate: 0.00%

Violation recall: 77.78%

Confusion matrix (rows actual, columns predicted):

| Actual    |   SAFE |   VIOLATION |
|:----------|-------:|------------:|
| SAFE      |     41 |           0 |
| VIOLATION |      2 |           7 |

Failure cases: 2. See zone_failure_cases.csv.



RESULTS VS MIDTERM TARGETS
Zone Accuracy: 96.00% (Target ≥90%) — MET
False Violation Rate: 0.00% (Target ≤10%) — MET
Violation Recall: 77.78% (Additional diagnostic metric)

Note: Results are calculated from the supplied labels.
Labels based on annotated video remain provisional until checked against original unannotated footage.


## E4. Download Evaluation Files

Download one ZIP containing `data/zone_labels_verified.csv` and the four files in `results/metrics/`. Extract the ZIP into matching GitHub folders. Keep the video-run export separately.


In [10]:
# ============================================================
# SECTION E4 — Package and Download Evaluation Results
# ============================================================

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
from datetime import datetime
import pandas as pd

# 1. Define project paths
REPO = Path("/content/PetGuard-AI")

metrics_dir = REPO / "results" / "metrics"
verified_path = REPO / "data" / "zone_labels_verified.csv"

# 2. Required evaluation files
evaluation_files = [
    "zone_evaluation.csv",
    "zone_confusion_matrix.csv",
    "zone_failure_cases.csv",
    "zone_evaluation_summary.md"
]

# 3. Verify that all files exist
required_paths = [
    verified_path,
    *[metrics_dir / name for name in evaluation_files]
]

missing = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

assert not missing, (
    "Cannot export. Missing files:\n" +
    "\n".join(missing)
)

# 4. Read evaluation information
labels = pd.read_csv(verified_path)
evaluation = pd.read_csv(
    metrics_dir / "zone_evaluation.csv"
)

assert not labels.empty, "Label CSV is empty."
assert not evaluation.empty, "Evaluation CSV is empty."

# 5. Create timestamped ZIP filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

zip_path = Path(
    f"/content/petguard_evaluation_{timestamp}.zip"
)

# 6. Package files with GitHub-compatible folder structure
with ZipFile(zip_path, "w", ZIP_DEFLATED) as z:

    # Original evaluation input
    z.write(
        verified_path,
        arcname="data/zone_labels_verified.csv"
    )

    # Generated evaluation outputs
    for name in evaluation_files:
        z.write(
            metrics_dir / name,
            arcname=f"results/metrics/{name}"
        )

# 7. Verify ZIP integrity
with ZipFile(zip_path, "r") as z:
    bad_file = z.testzip()
    assert bad_file is None, (
        f"ZIP integrity error: {bad_file}"
    )

# 8. Display summary
print("=" * 55)
print("PETGUARD AI — EVALUATION EXPORT")
print("=" * 55)

print(f"Samples evaluated : {len(labels)}")
print(f"ZIP created       : {zip_path.name}")
print(f"ZIP size          : {zip_path.stat().st_size / 1024:.1f} KB")

print("\nFiles included:")

with ZipFile(zip_path, "r") as z:
    for name in z.namelist():
        print("✓", name)

print("\nEvaluation summary:")

row = evaluation.iloc[0]

print(f"Zone accuracy        : {row['zone_accuracy']:.2%}")
print(f"False violation rate : {row['false_violation_rate']:.2%}")
print(f"Violation recall     : {row['violation_recall']:.2%}")

print("\nNote: Metrics reflect the supplied labels.")
print("Verify labels against original footage before")
print("claiming independently verified ground truth.")

# 9. Download ZIP to your PC
print("\nStarting download...")
files.download(str(zip_path))

PETGUARD AI — EVALUATION EXPORT
Samples evaluated : 50
ZIP created       : petguard_evaluation_20261113_004015.zip
ZIP size          : 1.9 KB

Files included:
✓ data/zone_labels_verified.csv
✓ results/metrics/zone_evaluation.csv
✓ results/metrics/zone_confusion_matrix.csv
✓ results/metrics/zone_failure_cases.csv
✓ results/metrics/zone_evaluation_summary.md

Evaluation summary:
Zone accuracy        : 96.00%
False violation rate : 0.00%
Violation recall     : 77.78%

Note: Metrics reflect the supplied labels.
Verify labels against original footage before
claiming independently verified ground truth.

Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>